# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema, provided by the following URL.

In [ ]:
# Ensure `mlcroissant` library is installed (uncomment if running in Colab or local environment)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print out the dataset's name and description
metadata_json = dataset.metadata.to_json()
print(f"Dataset name: {metadata_json['name']}")
print(f"Description: {metadata_json['description']}")

# Optionally, show the publication date and version
print(f"Date Published: {metadata_json.get('datePublished', 'N/A')}")
print(f"Version: {metadata_json.get('version', 'N/A')}")

## 2. Data Overview
The dataset may contain multiple record sets. Let's enumerate the available record sets, their `@id`s, and the `@id` fields for columns or attributes within those record sets.

We will reference all objects strictly by their `@id` as required.

In [ ]:
# List all available RecordSets and their field IDs

if hasattr(dataset, 'record_sets'):
    record_sets = list(dataset.record_sets)
else:
    # For older versions
    record_sets = [x['@id'] for x in dataset.metadata.to_json().get('recordSet', [])]  # fallback if available
print(f"Found {len(record_sets)} record sets:\n")
for rec_set in record_sets:
    print(f"RecordSet @id: {rec_set}")
    # List columns/fields for each record set
    try:
        columns = dataset.columns(record_set=rec_set)
        print("  Columns/Fields @id:")
        for col in columns:
            print(f"    - {col}")
    except Exception as e:
        print(f"  Could not retrieve fields for record set {rec_set}: {e}")
    print()
# Note: If there are no record sets found, check the Croissant schema file for available tables/files.

## 3. Data Extraction
We will select a record set (by `@id`) and load its records using `mlcroissant`. You can adapt the chosen `@id` or column `@id`s using the overview above.

For demonstration, we will attempt to load all available record sets into pandas DataFrames, referenced by their `@id`. If you know a specific table you'd like to explore, set its `@id` below.

In [ ]:
dataframes = {}

for rec_set in record_sets:
    try:
        records = list(dataset.records(record_set=rec_set))
        df = pd.DataFrame(records)
        dataframes[rec_set] = df
        print(f"Loaded {len(df)} records from RecordSet @id: {rec_set}")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for RecordSet {rec_set}: {e}")
    print()

# If record sets exist, pick the first one for further exploration
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"Using RecordSet @id for analysis: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Let's demonstrate typical EDA including value filtering, normalization, and grouping—all referencing columns by their `@id`.

You may adjust the column `@id`s below to analyze your field of interest based on available columns in the above cell.

In [ ]:
if main_record_set_id and not dataframes[main_record_set_id].empty:
    df = dataframes[main_record_set_id]
    # Select a numeric field for analysis by its @id - update as suited
    numeric_candidates = df.select_dtypes(include='number').columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Numeric field candidate for EDA: {numeric_field_id}")

        # Example threshold
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 10

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold} (showing up to 5):")
        display(filtered_df.head())

        # Normalization
        filtered_df[numeric_field_id + '_normalized'] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

        # Grouping by another column - try for a likely categorical field
        group_candidates = df.select_dtypes(include='object').columns.tolist()
        if group_candidates:
            group_field_id = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}: (showing up to 5 groups)")
            display(grouped_df.head())
        else:
            print("No suitable group field candidates found.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No records available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Adjust the field `@id`s used if more suitable variables are found.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and not dataframes[main_record_set_id].empty:
    df = dataframes[main_record_set_id]
    numeric_candidates = df.select_dtypes(include='number').columns.tolist()
    if numeric_candidates:
        field = numeric_candidates[0]
        plt.figure(figsize=(8, 5))
        sns.histplot(df[field], kde=True, bins=30)
        plt.title(f"Distribution of field '@id': {field}")
        plt.xlabel(field)
        plt.ylabel('Frequency')
        plt.show()
    else:
        print("No numeric field found for histogram visualization.")
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook demonstrated the basic workflow for loading, exploring, and analyzing a FAIR² mass spectrometry dataset using the Croissant format and the `mlcroissant` Python library.

You can extend this workflow by selecting more fields and exploring additional relationships, or by exporting processed tables for downstream tasks.